In [1]:
# Written for Python 3.12+ and beautifulsoup4 4.15.0

import csv
import re
import requests
from bs4 import BeautifulSoup

URL = "https://bolpatra.gov.np/egp/loadContractRecordsListPublic"

OUTPUT_FILE = "contract_records.csv"

HEADERS = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0 Safari/537.36",
  "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
  "Referer": "https://bolpatra.gov.np/"
}


def clean(text: str) -> str:
    return " ".join(text.split())


def extract_contract_id(action_td) -> str:
    """
    Extracts ID from:
    onclick="openViewContractRecordsPublic('IDO/CHT/W/NCB/05/078-79')"
    """
    a_tag = action_td.find("a", onclick=True)
    if not a_tag:
        return ""

    onclick = a_tag.get("onclick", "")
    match = re.search(r"openViewContractRecordsPublic\('(.+?)'\)", onclick)

    if not match:
        return ""

    return match.group(1).strip()


def scrape_contract_records():
    response = requests.get(URL, headers=HEADERS, timeout=30, verify=False)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    table = soup.find("table", id="projDetailsTab")
    if not table:
        raise RuntimeError("Could not find table with id='projDetailsTab'")

    rows = []

    for tr in table.select("tbody tr"):
        cells = tr.find_all("td")

        # Expected columns:
        # 0 SL No.
        # 1 PE Name
        # 2 Contract Name
        # 3 Contract Amount
        # 4 Procurement Category
        # 5 Procurement Method
        # 6 Status
        # 7 Action
        if len(cells) < 8:
            continue

        row = {
            "contract_id": extract_contract_id(cells[7]),
            "pe_name": clean(cells[1].get_text(" ", strip=True)),
            "contract_name": clean(cells[2].get_text(" ", strip=True)),
            "contract_amount": clean(cells[3].get_text(" ", strip=True)),
            "procurement_category": clean(cells[4].get_text(" ", strip=True)),
            "procurement_method": clean(cells[5].get_text(" ", strip=True)),
            "status": clean(cells[6].get_text(" ", strip=True)),
        }

        rows.append(row)

    return rows


def save_to_csv(rows, filename):
    fieldnames = [
        "contract_id",
        "pe_name",
        "contract_name",
        "contract_amount",
        "procurement_category",
        "procurement_method",
        "status",
    ]

    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


if __name__ == "__main__":
    records = scrape_contract_records()
    save_to_csv(records, OUTPUT_FILE)

    print(f"Saved {len(records)} records to {OUTPUT_FILE}")

/home/suchita/myenv/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'bolpatra.gov.np'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Saved 2650 records to contract_records.csv
